In [ ]:
import torch
import cv2
import numpy as np
import math
import matplotlib.pyplot as plt
import glob
import os
import pandas as pd
import segmentation_models_pytorch as smp 
import scipy.stats as stats
import time

begin = time.time()
# ================= CONFIGURATION =================
MODEL_PATH = "MODEL.pth" 
IMAGE_FOLDER = r"Images"
OUTPUT_CSV_NAME = "Analysis.csv"

MICRONS_PER_PIXEL =   ####INPUT 
SCALE_BAR_LENGTH_MICRONS = 1

PATCH_SIZE = 256
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ================= HELPER FUNCTIONS =================

def add_scale_bar(image, um_per_px, bar_length_um):
    h, w = image.shape[:2]
    bar_pixels = int(bar_length_um / um_per_px)
    margin = 40
    start_point = (w - margin - bar_pixels, h - margin)
    end_point = (w - margin, h - margin)
    
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    
    cv2.line(image, start_point, end_point, (0, 255, 0), 6)
    cv2.putText(image, f"{bar_length_um} um", (start_point[0], start_point[1] - 15), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    return image

def process_large_image(image_path, model, patch_size=256, stride=128):
    original_img = cv2.imread(image_path)
    if original_img is None: return None, None, None
    img_rgb = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    
    pad_h = (math.ceil(h / patch_size) * patch_size) - h
    pad_w = (math.ceil(w / patch_size) * patch_size) - w
    img_padded = cv2.copyMakeBorder(img_rgb, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)
    ph, pw = img_padded.shape[:2]
    
    full_mask_acc = np.zeros((ph, pw), dtype=np.float32)
    full_var_acc = np.zeros((ph, pw), dtype=np.float32)
    count_map = np.zeros((ph, pw), dtype=np.float32)
    
    with torch.no_grad():
        for y in range(0, ph - patch_size + 1, stride):
            for x in range(0, pw - patch_size + 1, stride):
                patch = img_padded[y:y+patch_size, x:x+patch_size]
                input_tensor = patch.transpose(2, 0, 1).astype('float32') / 255.0
                input_tensor = torch.from_numpy(input_tensor).unsqueeze(0).to(DEVICE)
                
                p1 = torch.sigmoid(model(input_tensor)).squeeze().cpu().numpy()
                logits_h = model(torch.flip(input_tensor, dims=[3]))
                p2 = torch.flip(torch.sigmoid(logits_h), dims=[3]).squeeze().cpu().numpy()
                logits_v = model(torch.flip(input_tensor, dims=[2]))
                p3 = torch.flip(torch.sigmoid(logits_v), dims=[2]).squeeze().cpu().numpy()
                logits_hv = model(torch.flip(input_tensor, dims=[2, 3]))
                p4 = torch.flip(torch.sigmoid(logits_hv), dims=[2, 3]).squeeze().cpu().numpy()
                
                stacked_probs = np.stack([p1, p2, p3, p4], axis=0)
                full_mask_acc[y:y+patch_size, x:x+patch_size] += np.mean(stacked_probs, axis=0)
                full_var_acc[y:y+patch_size, x:x+patch_size] += np.var(stacked_probs, axis=0)
                count_map[y:y+patch_size, x:x+patch_size] += 1.0

    final_mask_prob = full_mask_acc / np.maximum(count_map, 1)
    final_mask_binary = (final_mask_prob[:h, :w] > 0.5).astype(np.uint8) * 255
    final_var = (full_var_acc / np.maximum(count_map, 1))[:h, :w]
    overall_confidence = (1.0 - (np.mean(final_var) / 0.25)) * 100
    
    return original_img, final_mask_binary, overall_confidence

def analyze_particles(mask, um_per_px):
    """
    Analyzes connected components to calculate Equivalent Circle Diameter (ECD)
    and Aspect Ratio for each isolated particle in the mask.
    """
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    ecds = []
    aspect_ratios = []
    
    for cnt in contours:
        area_px = cv2.contourArea(cnt)
        if area_px < 3:  # Filter out microscopic noise (less than 3 pixels)
            continue
            
        # Equivalent Circle Diameter (ECD)
        area_um2 = area_px * (um_per_px ** 2)
        ecd = 2 * np.sqrt(area_um2 / np.pi)
        ecds.append(ecd)
        
        # Aspect Ratio (Using Minimum Area Bounding Rectangle)
        rect = cv2.minAreaRect(cnt)
        w, h = rect[1]
        if w == 0 or h == 0:
            continue
            
        ar = max(w, h) / min(w, h)
        aspect_ratios.append(ar)
        
    return ecds, aspect_ratios

def plot_and_save_histogram(data, title, xlabel, filename):
    plt.figure(figsize=(8, 6))
    plt.hist(data, bins=50, color='teal', edgecolor='black', alpha=0.7)
    plt.title(title, fontsize=14, pad=10)
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel("Frequency", fontsize=12)
    plt.grid(axis='y', alpha=0.4)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

# ================= MAIN EXECUTION =================

def main():
    print(f"Loading model on {DEVICE}...")
    model = smp.Unet(encoder_name="resnet18", encoder_weights=None, in_channels=3, classes=1, activation=None)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
    model.to(DEVICE).eval()
    
    image_files = []
    for ext in ["*.tif", "*.jpg", "*.png", "*.bmp"]:
        image_files.extend(glob.glob(os.path.join(IMAGE_FOLDER, ext)))
    image_files = [f for f in image_files if "_processed" not in f]
    
    print(f"Found {len(image_files)} images to process.")
    results_data = []
    
    # Global lists to hold all particle data across the entire dataset for plotting
    global_ecds = []
    global_aspect_ratios = []

    for idx, img_path in enumerate(image_files):
        filename = os.path.basename(img_path)
        print(f"[{idx+1}/{len(image_files)}] Processing {filename}...")
        
        original, mask, overall_confidence = process_large_image(img_path, model)
        if mask is None: continue 
            
        # A. Phase Fraction
        alpha_fraction = (cv2.countNonZero(mask) / (mask.shape[0] * mask.shape[1])) * 100
        
        # B. Particle Analysis
        ecds, aspect_ratios = analyze_particles(mask, MICRONS_PER_PIXEL)
        
        avg_ecd = np.mean(ecds) if len(ecds) > 0 else 0
        avg_ar = np.mean(aspect_ratios) if len(aspect_ratios) > 0 else 0
        
        global_ecds.extend(ecds)
        global_aspect_ratios.extend(aspect_ratios)

        # Saving Image Outputs
        base_name = os.path.splitext(filename)[0]
        mask_with_scale = add_scale_bar(mask.copy(), MICRONS_PER_PIXEL, SCALE_BAR_LENGTH_MICRONS)
        cv2.imwrite(os.path.join(IMAGE_FOLDER, f"{base_name}_processed.png"), mask_with_scale)
        
        # Collect Data
        results_data.append({
            "Filename": filename,
            "Overall Confidence (%)": round(overall_confidence, 2),
            "Phase Fraction (%)": round(alpha_fraction, 2),
            "Particle Count": len(ecds),
            "Avg Equivalent Circle Diameter (um)": round(avg_ecd, 4),
            "Avg Aspect Ratio": round(avg_ar, 4)
        })

    # C. Data Saving & Statistics
    if results_data:
        df = pd.DataFrame(results_data)
        df.to_csv(os.path.join(IMAGE_FOLDER, OUTPUT_CSV_NAME), index=False)
        
        n = len(df)
        if n > 1:
            t_val = stats.t.ppf(1 - 0.025, n - 1) 
            metrics = ['Phase Fraction (%)', 'Avg Equivalent Circle Diameter (um)', 'Avg Aspect Ratio']
            summary = []
            for m in metrics:
                mean_v, std_v = df[m].mean(), df[m].std(ddof=1)
                ci_95 = (t_val * std_v) / math.sqrt(n)
                ra = (ci_95 / mean_v) * 100 if mean_v > 0 else 0
                summary.append({
                    "Metric": m, 
                    "Avg": round(mean_v, 4), 
                    "Std Dev": round(std_v, 4), 
                    "95% CI": round(ci_95, 4), 
                    "Relative Accuracy%": round(ra, 2)
                })
            
            summary_df = pd.DataFrame(summary)
            summary_df.to_csv(os.path.join(IMAGE_FOLDER, "folder_summary_statistics.csv"), index=False)
            print("\n--- Folder Summary ---\n", summary_df.to_string(index=False))

    # D. Save Histograms
    print("\nGenerating histograms...")
    if global_ecds:
        plot_and_save_histogram(
            global_ecds, 
            "Distribution of Equivalent Circle Diameters", 
            "Equivalent Circle Diameter (um)", 
            os.path.join(IMAGE_FOLDER, "histogram_ECD.png")
        )
    if global_aspect_ratios:
        plot_and_save_histogram(
            global_aspect_ratios, 
            "Distribution of Aspect Ratios", 
            "Aspect Ratio", 
            os.path.join(IMAGE_FOLDER, "histogram_Aspect_Ratio.png")
        )
    print("Histograms saved successfully.")

if __name__ == "__main__":
    main()
    time.sleep(1)
    end = time.time()
    print (f"Total runtime of the program is {end-begin}")